In [ ]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
from itertools import chain, combinations
import warnings
warnings.simplefilter(action='ignore', category=DeprecationWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)
from pgmpy.estimators.BaseConstraintEstimator import BaseConstraintEstimator
%matplotlib inline
from itertools import product

import numpy as np
import pandas as pd
import networkx as nx
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.special import comb

from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    silhouette_score,
    pairwise_distances,
    jaccard_score,
    adjusted_rand_score,
    fowlkes_mallows_score,
)

from sklearn.metrics.cluster import (
    contingency_matrix,
    homogeneity_score,
    completeness_score,
    v_measure_score,
)


def plot_pca_matrix(X, y, dim=3):
    '''
    Builds a scatter matrix of the first `dim` principal components.
    Variance explained by each PC is shown in the axis label.
    Points are colored by cluster label y.
    '''
    pca = PCA()
    components = pca.fit_transform(X)
    labels = {
        str(i): f"PC {i+1} ({var:.1f}%)"
        for i, var in enumerate(pca.explained_variance_ratio_ * 100)
    }
    fig = px.scatter_matrix(
        components, labels=labels, dimensions=range(dim), color=y.astype(str)
    )
    fig.update_traces(diagonal_visible=False, showupperhalf=False)
    fig.update_layout(autosize=False, width=995, height=548)
    fig.show()



def plot_tsne(X, y):
    """
    Reduces X to 2D with t-SNE and plots the result as a scatter plot.
    Useful for visually inspecting cluster separability in a non-linear projection.
    """
    result = TSNE(n_components=2, random_state=0).fit_transform(X)

    df = pd.DataFrame({
        "tsne_1": result[:, 0],
        "tsne_2": result[:, 1],
        "label": y
    })

    # Ensure all cluster labels are represented
    labels = sorted(np.unique(y))

    # Larger figure
    fig, ax = plt.subplots(figsize=(10, 8))

    # Colorful palette with one color per label
    palette = sns.color_palette("husl", n_colors=len(labels))

    sns.scatterplot(
        x="tsne_1",
        y="tsne_2",
        hue="label",
        hue_order=labels,
        palette=palette,
        data=df,
        ax=ax,
        s=30,
        alpha=0.8
    )

    lim = (result.min() - 5, result.max() + 5)
    ax.set_xlim(lim)
    ax.set_ylim(lim)
    ax.set_aspect("equal")

    ax.legend(
        title="Cluster",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        borderaxespad=0
    )

    plt.tight_layout()
    plt.show()



def cluster_wss_bss(X, y_pred):
    '''
    Computes Within-Cluster Sum of Squares (WSS) and Between-Cluster Sum of Squares (BSS).
    BSS is derived as TSS - WSS, where TSS is the total variance of the dataset.
    Higher BSS relative to WSS indicates better-separated clusters.
    '''
    tss = ((X - X.mean(axis=0)) ** 2).sum()
    wss_val = 0.0
    for k in np.unique(y_pred):
        Xk = X[y_pred == k]
        wss_val += ((Xk - Xk.mean(axis=0)) ** 2).sum()
    return wss_val, tss - wss_val



def elbow_analysis(X, num_clus, method="kmeans", init="random", criterion="maxclust", linkage_method="ward", metric="euclidean"):
    '''
    Runs WSS/BSS and silhouette analysis for k from 1 to num_clus.
    Supports both kmeans and hierarchical clustering. For hierarchical,
    the linkage matrix is computed once and the dendrogram is cut at each k,
    which is more efficient than re-fitting from scratch.
    The resulting plots help identify the optimal number of clusters.
    '''
    wss_list, bss_list, sil_list = [], [], []
    clus_list = list(range(1, num_clus + 1))

    if method == "hierarchical":
        Z = linkage(X, method=linkage_method, metric=metric)

    for nc in clus_list:
        if method == "kmeans":
            model = KMeans(n_clusters=nc, init=init, random_state=0, n_init=10)
            y_pred = model.fit_predict(X)
        else:
            if nc == 1:
                y_pred = np.zeros(X.shape[0], dtype=int)
            else:
                y_pred = fcluster(Z, t=nc, criterion=criterion) - 1

        w, b = cluster_wss_bss(X, y_pred)
        wss_list.append(w)
        bss_list.append(b)
        if nc > 1:
            sil_list.append(silhouette_score(X, y_pred, metric="euclidean"))

    _, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(clus_list, wss_list, label="WSS")
    axes[0].plot(clus_list, bss_list, label="BSS")
    axes[0].set_title("WSS / BSS"); axes[0].legend()
    axes[1].plot(clus_list[1:], sil_list, label="Silhouette")
    axes[1].set_title("Average Silhouette Score"); axes[1].legend()
    plt.tight_layout(); plt.show()



def plot_dendrogram(Z, n_clusters=None, **kwargs):
    '''
    Plots a dendrogram from a precomputed linkage matrix Z.
    If n_clusters is specified, draws a horizontal threshold line that
    corresponds to cutting the tree into exactly that many clusters.
    '''
    fig, ax = plt.subplots(figsize=(20, 8))
    threshold = None
    if n_clusters is not None:
        threshold = Z[-(n_clusters), 2] + 1e-6
    dendrogram(Z, ax=ax, color_threshold=threshold,
               truncate_mode=kwargs.get("truncate_mode"),
               p=kwargs.get("p", 30))
    if threshold:
        ax.axhline(y=threshold, c="k", linestyle="--")
    plt.show()



def plot_sorted_similarity(X, y_pred, metric="euclidean"):
    '''
    Computes a pairwise similarity matrix from X and reorders rows/columns
    by cluster assignment. A good clustering shows dense blocks along the diagonal,
    indicating that points within the same cluster are mutually similar.
    '''
    dist = pairwise_distances(X, metric=metric)
    sim = 1 - (dist - dist.min()) / (dist.max() - dist.min())
    idx = np.argsort(y_pred)
    sim = sim[np.ix_(idx, idx)]
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(sim, ax=ax, xticklabels=False, yticklabels=False)
    plt.show()



def dbscan_grid_search(X, eps_values, min_samples_values,
                        noise_frac_max=0.4, cluster_range=(2, 20), verbose=True):
    """
    Manual grid search for DBSCAN hyperparameters, selecting eps and min_samples
    by silhouette score computed on the full dataset (no cross-validation:
    DBSCAN is not inductive, so train/test splitting doesn't apply).

    Noise points (label -1) are excluded from the silhouette computation,
    consistent with treating them as "absent" rather than a real cluster.

    Degenerate solutions (too few/many clusters, or too much noise) are
    filtered out before ranking, to avoid selecting trivial partitions that
    artificially maximize silhouette (e.g. one giant cluster + tiny outliers).

    """
    results = []
    for eps, min_samples in product(eps_values, min_samples_values):
        labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(X)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        mask = labels != -1
        noise_frac = (labels == -1).mean()

        if n_clusters < 2 or mask.sum() < 2:
            continue

        score = silhouette_score(X[mask], labels[mask])
        results.append({
            "eps": eps, "min_samples": min_samples,
            "silhouette": score, "n_clusters": n_clusters,
            "noise_frac": noise_frac
        })

    grid_df = pd.DataFrame(results).sort_values("silhouette", ascending=False)

    grid_df_valid = grid_df[
        (grid_df["noise_frac"] < noise_frac_max) &
        (grid_df["n_clusters"].between(*cluster_range))
    ]

    if grid_df_valid.empty:
        raise ValueError(
            "No valid configuration found within the given constraints "
            f"(noise_frac < {noise_frac_max}, n_clusters in {cluster_range}). "
            "Widen the eps/min_samples ranges or relax the constraints."
        )

    best_row = grid_df_valid.sort_values("silhouette", ascending=False).iloc[0]
    best_model = DBSCAN(eps=best_row["eps"], min_samples=int(best_row["min_samples"])).fit(X)

    if verbose:
        print(f"Best params: eps={best_row['eps']:.2f}, "
              f"min_samples={int(best_row['min_samples'])}, "
              f"silhouette={best_row['silhouette']:.3f}, "
              f"n_clusters={best_row['n_clusters']}, "
              f"noise={best_row['noise_frac']:.1%}")

    return best_model, grid_df_valid.sort_values("silhouette", ascending=False)



def tot_purity(y_pred, y_true):
    '''
    Computes clustering purity: for each predicted cluster, counts the majority
    true class and sums those counts over the total number of (non-noise) points.
    Noise points (label == -1) are excluded before computation.
    '''
    mask = y_pred != -1
    y_pred, y_true = y_pred[mask], y_true[mask]
    cm = contingency_matrix(y_true, y_pred)
    return cm.max(axis=0).sum() / y_true.shape[0]



def print_external_metrics(y_pred, y_true, label=""):
    '''
    Prints a summary of external clustering metrics against ground truth labels:
    purity, homogeneity, completeness, V-measure, ARI, Jaccard pairs, FM, and Hubert's Gamma.
    '''
    mask = y_pred != -1
    y_pred_filtered, y_true_filtered = y_pred[mask], y_true[mask]

    if len(y_pred_filtered) == 0:
        print(f"--- {label} --- Error: No valid points (only noise).")
        return

    # Information-theoretic and purity metrics
    cm = contingency_matrix(y_true_filtered, y_pred_filtered)
    purity = cm.max(axis=0).sum() / y_true_filtered.shape[0]
    homo = homogeneity_score(y_true_filtered, y_pred_filtered)
    comp = completeness_score(y_true_filtered, y_pred_filtered)
    v_med = v_measure_score(y_true_filtered, y_pred_filtered)

    # Pair-based metrics calculations
    ari = adjusted_rand_score(y_true_filtered, y_pred_filtered)
    fm = fowlkes_mallows_score(y_true_filtered, y_pred_filtered)

    tp_plus_fp = sum(comb(n, 2) for n in cm.sum(axis=0))
    tp_plus_fn = sum(comb(n, 2) for n in cm.sum(axis=1))
    tp = sum(sum(comb(n, 2) for n in row) for row in cm)
    fp = tp_plus_fp - tp
    fn = tp_plus_fn - tp

    jaccard_pairs = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0

    total_pairs = comb(len(y_true_filtered), 2)
    denom = np.sqrt(tp_plus_fp * tp_plus_fn * (total_pairs - tp_plus_fp) * (total_pairs - tp_plus_fn))
    hubert_gamma = (total_pairs * tp - tp_plus_fp * tp_plus_fn) / denom if denom > 0 else 0.0

    print(f"--- {label} ---")
    print(f"  Adjusted Rand Index (ARI):     {ari:.4f}")
    print(f"  Jaccard (pairs):               {jaccard_pairs:.4f}")
    print(f"  Fowlkes-Mallows (FM):          {fm:.4f}")
    print(f"  Hubert's Gamma Statistic:      {hubert_gamma:.4f}")
    print(f"  Purity:                        {purity:.4f}")
    print(f"  Homogeneity:                   {homo:.4f}")
    print(f"  Completeness:                  {comp:.4f}")
    print(f"  V-measure:                     {v_med:.4f}")




def clustering_comparison_table(results, X, y):
    '''
    Builds a summary DataFrame comparing multiple clustering solutions.
    results: dict of {"method name": y_pred array}
    Combines internal metrics (Silhouette, WSS, BSS) and external metrics
    (ARI, FM, Purity, Homogeneity, Completeness, V-measure).
    '''
    rows = []
    for name, y_pred in results.items():
        mask = y_pred != -1
        y_pred_f = y_pred[mask]
        y_true_f = y[mask]

        n_clusters = len(np.unique(y_pred_f))
        n_noise    = (y_pred == -1).sum()

        sil      = silhouette_score(X[mask], y_pred_f) if n_clusters > 1 else np.nan
        wss, bss = cluster_wss_bss(X[mask], y_pred_f)

        cm     = contingency_matrix(y_true_f, y_pred_f)
        purity = cm.max(axis=0).sum() / y_true_f.shape[0]
        ari    = adjusted_rand_score(y_true_f, y_pred_f)
        fm     = fowlkes_mallows_score(y_true_f, y_pred_f)
        homo   = homogeneity_score(y_true_f, y_pred_f)
        comp   = completeness_score(y_true_f, y_pred_f)
        vmes   = v_measure_score(y_true_f, y_pred_f)

        rows.append({
            "Method":       name,
            "k":            n_clusters,
            "Noise pts":    n_noise,
            "Silhouette":   round(sil,    4),
            "WSS":          round(wss,    2),
            "BSS":          round(bss,    2),
            "ARI":          round(ari,    4),
            "FM":           round(fm,     4),
            "Purity":       round(purity, 4),
            "Homogeneity":  round(homo,   4),
            "Completeness": round(comp,   4),
            "V-measure":    round(vmes,   4),
        })

    return pd.DataFrame(rows).set_index("Method")



def analyze_ordinal_variables(df, cols):
    """
    Plots the value distribution of variables that could be treated
    as either ordinal or continuous. Prints distinct level counts
    to support the encoding decision: with 6+ levels and roughly
    uniform spacing, treating them as continuous is defensible.
    """
    fig, axes = plt.subplots(1, len(cols), figsize=(6 * len(cols), 4))
    if len(cols) == 1:
        axes = [axes]

    for ax, col in zip(axes, cols):
        counts = df[col].value_counts().sort_index()
        ax.bar(counts.index, counts.values)
        ax.set_xlabel(col)
        ax.set_ylabel("Count")
        ax.set_xticks(counts.index)
        ax.set_title(f"{col} — {counts.shape[0]} distinct levels")

    plt.tight_layout()
    plt.show()

    for col in cols:
        counts = df[col].value_counts().sort_index()
        print(f"\n{col}:")
        print(counts.to_string())
        print(f"  Distinct levels : {counts.shape[0]}")
        print(f"  Range           : {df[col].min()} – {df[col].max()}")



def encoding_sensitivity_check(df, numeric_vars, ordinal_cols, threshold=0.05):
    """
    Compares Pearson and Spearman correlations for selected ordinal variables.
    Small differences between the two correlation measures suggest that
    treating the ordinal variables as numeric is unlikely to substantially
    alter the observed correlation structure. This provides empirical support
    for retaining the variables in their numeric form before scaling with MinMaxScaler().
    """
    # Keep only columns that are actually numeric in the dataframe
    valid_vars = [c for c in numeric_vars if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]

    pearson  = df[valid_vars].corr(method="pearson")
    spearman = df[valid_vars].corr(method="spearman")
    diff     = (spearman - pearson).abs()

    print("Max absolute difference Spearman vs Pearson:")
    for col in ordinal_cols:
        max_diff = diff[col].max()
        verdict  = "✓ continuous treatment justified" if max_diff < threshold else "⚠ consider ordinal encoding"
        print(f"  {col}: {max_diff:.4f}  — {verdict}")

    return diff



def detect_outliers_grubbs(X, feature_names, alpha=0.05, exclude_binary=True, verbose=True):
    """
    Multivariate outlier detection via the iterative Grubbs test (generalized
    ESD), applied independently to each feature.

    For each feature, the test statistic is:
        G = max_i |x_i - mean(x)| / std(x)
    compared against a critical value derived from the t-distribution:
        G_crit = (n-1)/sqrt(n) * sqrt(t^2 / (n-2+t^2)),  t = t_(alpha/(2n), n-2)

    If G > G_crit, the most extreme point is flagged, removed from the
    working sample, and the test is repeated on the remaining points
    (this is what makes it "iterative" / ESD instead of single-point Grubbs).
    The loop stops as soon as G <= G_crit.

    If exclude_binary=True, columns with only {0, 1} values are skipped,
    for the same reason as in the Z-score version: Grubbs assumes
    approximate normality, which doesn't hold for one-hot/binary columns.

    Returns a boolean mask (True = outlier in at least one feature).
    """
    from scipy import stats

    feature_names = list(feature_names)
    n_total = X.shape[0]

    if exclude_binary:
        numeric_mask = np.array([
            len(np.unique(X[:, i])) > 2
            for i in range(X.shape[1])
        ])
        used_idx = np.where(numeric_mask)[0]
        used_features = [feature_names[i] for i in used_idx]
        if verbose:
            print(f"Grubbs applied to {len(used_idx)} non-binary features "
                  f"(skipping {X.shape[1] - len(used_idx)} binary/one-hot columns)")
    else:
        used_idx = np.arange(X.shape[1])
        used_features = feature_names

    outlier_mask = np.zeros(n_total, dtype=bool)
    triggered = {}

    for col_i, name in zip(used_idx, used_features):
        x = X[:, col_i]
        active_idx = np.arange(n_total)
        flagged_this_feature = 0

        while True:
            n = len(active_idx)
            if n < 3:
                break
            x_active = x[active_idx]
            mean, std = x_active.mean(), x_active.std()
            if std == 0:
                break

            abs_dev = np.abs(x_active - mean)
            worst_local = np.argmax(abs_dev)
            G = abs_dev[worst_local] / std

            t_crit = stats.t.ppf(1 - alpha / (2 * n), n - 2)
            G_crit = ((n - 1) / np.sqrt(n)) * np.sqrt(t_crit ** 2 / (n - 2 + t_crit ** 2))

            if G > G_crit:
                global_idx = active_idx[worst_local]
                outlier_mask[global_idx] = True
                flagged_this_feature += 1
                active_idx = np.delete(active_idx, worst_local)
            else:
                break

        if flagged_this_feature:
            triggered[name] = flagged_this_feature

    if verbose:
        print(f"Grubbs alpha      : {alpha}")
        print(f"Outliers flagged  : {outlier_mask.sum()} / {n_total} "
              f"({outlier_mask.mean() * 100:.1f}%)")
        if triggered:
            print("\nFeatures triggering outlier flags:")
            print(pd.Series(triggered).sort_values(ascending=False).to_string())

    return outlier_mask



def detect_outliers_lof(X, n_neighbors=20, contamination=0.05):
    """
    Outlier detection via Local Outlier Factor (Breunig et al., 2000).
    Compares the local density of a point to the local density of its
    n_neighbors nearest neighbors: points whose neighborhood is much
    denser than their own are flagged as outliers. Unlike Isolation
    Forest, this is explicitly density-based, which makes it consistent
    with DBSCAN's notion of density used later for clustering.
    `contamination` sets the expected fraction of outliers.
    Returns a boolean mask (True = outlier).
    """
    from sklearn.neighbors import LocalOutlierFactor

    lof = LocalOutlierFactor(n_neighbors=n_neighbors, contamination=contamination)
    preds = lof.fit_predict(X)
    outlier_mask = preds == -1

    print(f"n_neighbors       : {n_neighbors}")
    print(f"Contamination     : {contamination}")
    print(f"Outliers flagged  : {outlier_mask.sum()} / {len(X)} "
          f"({outlier_mask.mean() * 100:.1f}%)")

    return outlier_mask



def plot_outliers_pca(X, mask_a, mask_b, label_a="Grubbs", label_b="LOF"):
    """
    Projects X onto its first two principal components and plots three panels:
    - mask_a outliers vs inliers
    - mask_b outliers vs inliers
    - agreement / disagreement between the two methods
    Useful for visually assessing whether the two methods converge on the
    same regions of the feature space.
    """
    pca = PCA(n_components=2)
    X2 = pca.fit_transform(X)
    ev = pca.explained_variance_ratio_

    both     = mask_a & mask_b
    either   = mask_a | mask_b
    disagree = either & ~both

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    titles = [label_a, label_b, "Agreement / Disagreement"]
    masks_pairs = [mask_a, mask_b, None]

    for i, ax in enumerate(axes):
        if i < 2:
            mask = masks_pairs[i]
            ax.scatter(X2[~mask, 0], X2[~mask, 1], s=15, alpha=0.4, label="inlier")
            ax.scatter(X2[mask, 0], X2[mask, 1], s=30, alpha=0.8, label="outlier", marker="x")
        else:
            ax.scatter(X2[~either, 0], X2[~either, 1], s=15, alpha=0.3, label="inlier (both)")
            ax.scatter(X2[both, 0], X2[both, 1], s=40, alpha=0.9, label="both flag", marker="x")
            ax.scatter(X2[disagree, 0], X2[disagree, 1], s=40, alpha=0.7, label="one flag only", marker="^")

        ax.set_xlabel(f"PC1 ({ev[0]:.1%})")
        ax.set_ylabel(f"PC2 ({ev[1]:.1%})")
        ax.set_title(titles[i])
        ax.legend(fontsize=8)

    plt.tight_layout()
    plt.show()

    print(f"\nFlagged by both methods : {both.sum()}")
    print(f"Flagged by {label_a} only : {(mask_a & ~mask_b).sum()}")
    print(f"Flagged by {label_b} only : {(mask_b & ~mask_a).sum()}")



def dbscan_eps_knee(X, min_samples, metric="euclidean", S=1, curve="convex",
                     direction="increasing", plot=True, title=None):
    """
    Estimates a candidate eps for DBSCAN using the knee (elbow) method on the
    k-distance graph, independently of silhouette or any label-based score.

    For each point, computes the distance to its min_samples-th nearest neighbor
    (min_samples plays the role of MinPts here and is treated as fixed, not
    searched). Points inside real clusters have this distance small; points in
    low-density regions have it large, so sorting these distances and locating
    the knee gives a purely geometric candidate for eps.

    Returns eps_knee (float).
    """
    from sklearn.neighbors import NearestNeighbors
    from kneed import KneeLocator

    nbrs = NearestNeighbors(n_neighbors=min_samples, metric=metric).fit(X)
    distances, _ = nbrs.kneighbors(X)
    distances = np.sort(distances[:, min_samples - 1])

    idx = np.arange(len(distances))
    knee = KneeLocator(idx, distances, S=S, curve=curve,
                        direction=direction, interp_method="polynomial")

    eps_knee = knee.knee_y if knee.knee_y is not None else float(distances[-1])

    if plot:
        fig, ax = plt.subplots(figsize=(6, 5))
        ax.plot(idx, distances)
        if knee.knee is not None:
            ax.axvline(x=knee.knee, color="k", linestyle="--")
            ax.axhline(y=eps_knee, color="k", linestyle="--")
            ax.plot(knee.knee, eps_knee, "o", color="r")
        ax.set_xlabel("Points (sorted)")
        ax.set_ylabel(f"{min_samples}-th Nearest Neighbour Distance")
        ax.set_title(title or f"Knee method (min_samples={min_samples})")
        ax.grid(True)
        plt.tight_layout()
        plt.show()

    print(f"min_samples (fixed) : {min_samples}")
    print(f"eps (knee estimate) : {eps_knee:.3f}")

    return eps_knee



def dbscan_grid_search_narrow(X, eps_center, min_samples_values, eps_window=0.3,
                               eps_step=0.05, noise_frac_max=0.4,
                               cluster_range=(2, 20), verbose=True):
    """
    Same selection logic as dbscan_grid_search (silhouette on non-noise points,
    with degenerate solutions filtered out), but restricted to an eps range
    centered on a knee-based estimate (eps_center) instead of an arbitrary wide
    range. Narrowing eps this way means the compared configurations differ
    mostly in min_samples rather than spanning wildly different noise regimes,
    which reduces the risk of silhouette rewarding configurations simply
    because they discard more points as noise.
    """
    eps_values = np.arange(max(eps_center - eps_window, 1e-3),
                            eps_center + eps_window, eps_step)

    return dbscan_grid_search(X, eps_values, min_samples_values,
                               noise_frac_max=noise_frac_max,
                               cluster_range=cluster_range, verbose=verbose)



def dbscan_conservative_pick(grid_df, tol=0.01):
    """
    Among the configurations whose silhouette lies within `tol` of the best one,
    returns the one with the lowest noise fraction instead of the raw argmax.
    Silhouette computed on non-noise points can reward configurations that simply
    discard more points, so this prefers the most conservative configuration among
    the statistically equivalent ones.
    """
    best_score = grid_df["silhouette"].max()
    candidates = grid_df[grid_df["silhouette"] >= best_score - tol]
    return candidates.sort_values("noise_frac").iloc[0]



def prepare_bn_data(df, target, n_bins=3):
    """
    Prepares a DataFrame for Bayesian Network learning by discretising
    continuous columns into n_bins equal-frequency bins (tertiles by default).
    Binary columns (only 0/1) and already low-cardinality integer columns
    (≤ n_bins unique values) are kept as-is.

    Returns:
        bn_df   : DataFrame with integer-coded discrete columns
        disc_maps: dict {col: bin_edges} for continuous columns that were binned,
                   useful for mapping new evidence values to the correct bin index.
    """

    bn_df = df.copy().reset_index(drop=True)
    disc_maps = {}

    for col in bn_df.columns:
        series = bn_df[col]
        unique_vals = series.nunique()

        # Already discrete enough or binary → keep as integer
        if unique_vals <= n_bins or set(series.unique()).issubset({0, 1}):
            bn_df[col] = series.astype(int)
            continue

        # Continuous → quantile-based binning into n_bins levels (0, 1, …, n_bins-1)
        binned, bin_edges = pd.qcut(
            series, q=n_bins, labels=False,
            duplicates="drop", retbins=True
        )
        bn_df[col] = binned.astype(int)
        disc_maps[col] = bin_edges

    return bn_df, disc_maps

def _legal_operations_sorted(self, *args, **kwargs):
    return sorted(_orig_legal_ops(self, *args, **kwargs), key=lambda op: str(op))


def _get_potential_sepsets_sorted(u, v, temporal_ordering, graph, lim_neighbors):
    separating_set_u = set(graph.neighbors(u))
    separating_set_v = set(graph.neighbors(v))
    separating_set_u.discard(v)
    separating_set_v.discard(u)

    if temporal_ordering != dict():
        max_order = min(temporal_ordering[u], temporal_ordering[v])
        for neigh in list(separating_set_u):
            if temporal_ordering[neigh] > max_order:
                separating_set_u.discard(neigh)
        for neigh in list(separating_set_v):
            if temporal_ordering[neigh] > max_order:
                separating_set_v.discard(neigh)

    return chain(
        combinations(sorted(separating_set_u, key=str), lim_neighbors),
        combinations(sorted(separating_set_v, key=str), lim_neighbors),
    )

